# Lab：MindIE / AISBench 并发压测与图表绘制

## 1. 实验介绍

本实验使用 AISBench 对已经启动的 MindIE 推理服务执行并发压测，汇总每个并发点的吞吐量与端到端延迟，并输出 CSV 和三张性能曲线。

运行结束后应得到：

- <code>concurrency_results_raw.csv</code>：每一正式轮的原始指标；
- <code>concurrency_results.csv</code>：按并发数聚合后的指标；
- 并发数—吞吐量、并发数—P95/P99 E2EL、吞吐量—P99 E2EL 三张图。

本 Notebook 参考课程实验的“说明—执行—验收”形式编排。它不负责安装环境或启动 MindIE；开始前请先按服务器上的 <code>/home/ma-user/AGENT.md</code> 启动并验证服务。压测期间只改变 AISBench 的 <code>batch_size</code>，不修改 MindIE 服务配置。


## 2. 压测方法与约束

固定条件：

- 模型：Qwen2-7B，接口 <code>http://127.0.0.1:1025</code>；
- 数据集：GSM8K 4-shot CoT chat；
- 输出：<code>max_out_len=512</code>、<code>ignore_eos=True</code>；
- 发压：<code>request_rate=0</code>，仅改变 <code>batch_size</code>；
- 并发点：1、2、4、8、16、32、64、128、192、256；
- 每个并发点预热 1 轮、64 个请求；
- 并发小于 16 时正式测量 1 轮；并发不小于 16 时计划测量 3 轮；
- 每轮正式请求数为 <code>min(1319, max(128, batch_size × 8))</code>；
- 如果某个高并发点首轮相对上一个完整点出现吞吐回落，且 P99 增长至少 25%，则记录为过载点并停止后续轮次。

GSM8K 测试集只有 1319 条唯一题目，因此 192 和 256 并发的请求数会受数据集上限约束。本实验不重复题目，以免改变输入分布。


In [ ]:
import csv
import json
import os
import re
import shutil
import socket
import statistics
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

lab = {}


## 3. 路径和实验参数

下面的路径与本实验的服务器备份一致。若服务器上的 AISBench 或实验目录不同，只需修改这一单元格。Notebook 通过 AISBench 虚拟环境中的绝对路径启动命令，不会在运行中安装或升级依赖。


In [ ]:
BENCH_REPO = Path("/home/ma-user/work/aisbench-benchmark")
BENCH_BIN = Path("/home/ma-user/work/aisbench-venv311/bin/ais_bench")
EXP_ROOT = Path("/home/ma-user/work/mindie-concurrency-benchmark")

CONFIG_DIR = EXP_ROOT / "config"
CONFIG_FILE = CONFIG_DIR / "aisbench_qwen2_gsm8k.py"
OUTPUT_ROOT = EXP_ROOT / "outputs" / "formal"
PROGRESS_FILE = EXP_ROOT / "progress.csv"
FINAL_DIR = EXP_ROOT / "final"
SUMMARY_CSV = FINAL_DIR / "concurrency_results.csv"
RAW_CSV = FINAL_DIR / "concurrency_results_raw.csv"
CHART_DIR = FINAL_DIR / "charts"

SERVICE_HOST = "127.0.0.1"
SERVICE_PORT = 1025
MODEL_PATH = Path("/home/ma-user/models/Qwen2-7B")

CONCURRENCIES = [1, 2, 4, 8, 16, 32, 64, 128, 192, 256]
WARMUP_PROMPTS = 64
MIN_STABLE_PROMPTS = 128
DATASET_LIMIT = 1319
OVERLOAD_P99_GROWTH = 1.25

for directory in [CONFIG_DIR, OUTPUT_ROOT, FINAL_DIR, CHART_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("实验目录:", EXP_ROOT)
print("AISBench:", BENCH_BIN)
print("服务地址:", f"http://{SERVICE_HOST}:{SERVICE_PORT}")


## 4. 启动前检查

本单元格只读取环境状态。检查通过后才执行压测，避免在服务未就绪、路径错误或 AISBench 不存在时长时间等待。


In [ ]:
def port_is_open(host, port, timeout=3):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


checks = {
    "AISBench 仓库": BENCH_REPO.is_dir(),
    "AISBench 命令": BENCH_BIN.is_file() and os.access(BENCH_BIN, os.X_OK),
    "模型目录": MODEL_PATH.is_dir(),
    "MindIE 1025 端口": port_is_open(SERVICE_HOST, SERVICE_PORT),
}
lab["preflight"] = checks

for name, ok in checks.items():
    print(f"[{'OK' if ok else 'FAIL'}] {name}")

if not all(checks.values()):
    failed = [name for name, ok in checks.items() if not ok]
    raise RuntimeError("启动前检查未通过：" + "、".join(failed))

version_result = subprocess.run(
    [str(BENCH_BIN), "--version"],
    cwd=BENCH_REPO,
    text=True,
    capture_output=True,
    timeout=30,
)
print((version_result.stdout + version_result.stderr).strip()[:2000])


## 5. 生成 AISBench 配置

配置与历史实验保持一致，使用 AISBench 的流式自定义 API 模型适配器访问 MindIE。每个并发点开始前会替换模板中的并发数并写入配置文件。


In [ ]:
CONFIG_TEMPLATE = r'''from mmengine.config import read_base

from ais_bench.benchmark.calculators import DefaultPerfMetricCalculator
from ais_bench.benchmark.models import VLLMCustomAPIChatStream
from ais_bench.benchmark.partitioners import NaivePartitioner
from ais_bench.benchmark.runners.local_api import LocalAPIRunner
from ais_bench.benchmark.summarizers import DefaultPerfSummarizer
from ais_bench.benchmark.tasks import OpenICLInferTask
from ais_bench.benchmark.utils.model_postprocessors import extract_non_reasoning_content

with read_base():
    from ais_bench.benchmark.configs.datasets.gsm8k.gsm8k_gen_4_shot_cot_chat_prompt import gsm8k_datasets

models = [
    dict(
        attr="service",
        type=VLLMCustomAPIChatStream,
        abbr="qwen2-7b",
        path="/home/ma-user/models/Qwen2-7B",
        model="qwen2-7b",
        request_rate=0,
        retry=2,
        host_ip="127.0.0.1",
        host_port=1025,
        max_seq_len=2560,
        max_out_len=512,
        batch_size=__BATCH_SIZE__,
        trust_remote_code=False,
        generation_kwargs=dict(
            temperature=0.5,
            top_k=10,
            top_p=0.95,
            seed=None,
            repetition_penalty=1.03,
            ignore_eos=True,
        ),
        pred_postprocessor=dict(type=extract_non_reasoning_content),
    )
]

datasets = [*gsm8k_datasets]

summarizer = dict(
    type=DefaultPerfSummarizer,
    calculator=dict(
        type=DefaultPerfMetricCalculator,
        stats_list=["Average", "Min", "Max", "Median", "P75", "P90", "P95", "P99"],
    ),
)

infer = dict(
    partitioner=dict(type=NaivePartitioner),
    runner=dict(
        type=LocalAPIRunner,
        max_num_workers=1,
        task=dict(type=OpenICLInferTask),
    ),
)
'''


def write_aisbench_config(concurrency):
    if concurrency not in CONCURRENCIES:
        raise ValueError(f"未定义的并发数：{concurrency}")
    CONFIG_FILE.write_text(
        CONFIG_TEMPLATE.replace("__BATCH_SIZE__", str(concurrency)),
        encoding="utf-8",
    )
    return CONFIG_FILE


write_aisbench_config(CONCURRENCIES[0])
print(CONFIG_FILE)


## 6. 查看测试计划

表中“计划正式轮数”是正常情况下的轮数。检测到过载后，当前点可以只保留首轮，并停止后续并发测试。


In [ ]:
def stable_runs_for(concurrency):
    return 1 if concurrency < 16 else 3


def stable_prompts_for(concurrency):
    return min(DATASET_LIMIT, max(MIN_STABLE_PROMPTS, concurrency * 8))


plan_df = pd.DataFrame(
    [
        {
            "并发数": concurrency,
            "预热请求数": WARMUP_PROMPTS,
            "计划正式轮数": stable_runs_for(concurrency),
            "每轮正式请求数": stable_prompts_for(concurrency),
        }
        for concurrency in CONCURRENCIES
    ]
)
display(plan_df)


## 7. AISBench 结果读取函数

AISBench 每次运行会在工作目录中创建时间戳子目录。本节负责定位结果、读取请求吞吐量、E2EL 分位数和逐请求延迟，并用 <code>COMPLETE</code> 标记支持断点续跑。


In [ ]:
def parse_number(value):
    if isinstance(value, (int, float)):
        return float(value)
    match = re.search(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)", str(value))
    if not match:
        raise ValueError(f"无法从 {value!r} 中提取数值")
    return float(match.group(0))


def read_metric_csv(path):
    with path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    return {row["Performance Parameters"]: row for row in rows}


def latest_result_dir(run_root):
    candidates = sorted(
        path for path in run_root.glob("20*")
        if path.is_dir()
    )
    return candidates[-1] if candidates else None


def completed_result_dir(run_root):
    marker = run_root / "COMPLETE"
    if not marker.is_file():
        return None
    result_dir = Path(marker.read_text(encoding="utf-8").strip())
    return result_dir if result_dir.is_dir() else None


def result_paths(result_dir):
    perf_dir = result_dir / "performances" / "qwen2-7b"
    return {
        "common": perf_dir / "gsm8kdataset.json",
        "metrics": perf_dir / "gsm8kdataset.csv",
        "details": perf_dir / "gsm8kdataset_details.json",
    }


def read_run_result(result_dir):
    paths = result_paths(result_dir)
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError("AISBench 结果文件不完整：" + "、".join(missing))

    common = json.loads(paths["common"].read_text(encoding="utf-8"))
    metrics = read_metric_csv(paths["metrics"])
    details = json.loads(paths["details"].read_text(encoding="utf-8"))
    e2el = metrics["E2EL"]

    return {
        "result_dir": result_dir,
        "request_count": int(common["Total Requests"]["total"]),
        "success_requests": int(common["Success Requests"]["total"]),
        "failed_requests": int(common["Failed Requests"]["total"]),
        "throughput_requests_s": parse_number(common["Request Throughput"]["total"]),
        "avg_latency_ms": parse_number(e2el["Average"]),
        "p95_latency_ms": parse_number(e2el["P95"]),
        "p99_latency_ms": parse_number(e2el["P99"]),
        "output_token_throughput_tokens_s": parse_number(
            common["Output Token Throughput"]["total"]
        ),
        "request_latencies_ms": [
            float(value) for value in details["requests"]["seq_latency"]
        ],
    }


## 8. 执行单轮压测

AISBench 的实际调用形式为：

<code>ais_bench CONFIG --num-prompts N --mode perf --work-dir RUN_ROOT</code>

标准输出和错误输出写入每轮目录下的 <code>aisbench.log</code>。只有命令返回码为 0、结果文件存在且失败请求数为 0 时才写入完成标记。


In [ ]:
def append_progress(concurrency, phase, status, exit_code, result_dir):
    new_file = not PROGRESS_FILE.exists()
    with PROGRESS_FILE.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        if new_file:
            writer.writerow(
                ["concurrency", "phase", "status", "exit_code", "result_dir"]
            )
        writer.writerow(
            [concurrency, phase, status, exit_code, str(result_dir or "")]
        )


def run_aisbench(concurrency, phase, num_prompts):
    run_root = OUTPUT_ROOT / f"c{concurrency}" / phase
    run_root.mkdir(parents=True, exist_ok=True)
    log_file = run_root / "aisbench.log"

    existing = completed_result_dir(run_root)
    if existing is not None:
        print(f"[SKIP] c={concurrency} {phase} 已完成")
        return read_run_result(existing)

    write_aisbench_config(concurrency)
    command = [
        str(BENCH_BIN),
        str(CONFIG_FILE),
        "--num-prompts",
        str(num_prompts),
        "--mode",
        "perf",
        "--work-dir",
        str(run_root),
    ]
    print(f"[RUN] c={concurrency} {phase} requests={num_prompts}")

    with log_file.open("w", encoding="utf-8") as log_handle:
        completed = subprocess.run(
            command,
            cwd=BENCH_REPO,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            check=False,
        )

    result_dir = latest_result_dir(run_root)
    status = "failed"
    run_result = None
    if completed.returncode == 0 and result_dir is not None:
        run_result = read_run_result(result_dir)
        if run_result["failed_requests"] == 0:
            status = "complete"
            (run_root / "COMPLETE").write_text(
                str(result_dir) + "\n",
                encoding="utf-8",
            )

    append_progress(
        concurrency,
        phase,
        status,
        completed.returncode,
        result_dir,
    )
    if status != "complete":
        tail = log_file.read_text(encoding="utf-8", errors="replace")[-4000:]
        raise RuntimeError(
            f"c={concurrency} {phase} 未完成，日志末尾：\n{tail}"
        )

    return run_result


## 9. 执行完整并发扫描

下面的单元格会真正启动 AISBench 压测，运行时间可能较长。它会先做预热，再执行正式轮次；已有 <code>COMPLETE</code> 标记的轮次会自动跳过。

过载判断只用于决定是否停止继续发压，不会在图中添加任何说明或特殊标记。


In [ ]:
def pooled_point_result(run_results):
    latencies = [
        latency
        for run_result in run_results
        for latency in run_result["request_latencies_ms"]
    ]
    return {
        "throughput_requests_s": statistics.fmean(
            result["throughput_requests_s"] for result in run_results
        ),
        "p99_latency_ms": float(np.percentile(latencies, 99)),
    }


def run_sweep():
    previous_point = None

    for concurrency in CONCURRENCIES:
        overload_marker = OUTPUT_ROOT / f"c{concurrency}" / "OVERLOAD_STOP"
        if overload_marker.exists():
            print(f"[STOP] 已存在并发 {concurrency} 的过载停止标记")
            return {"status": "overload_stop", "concurrency": concurrency}

        if not port_is_open(SERVICE_HOST, SERVICE_PORT):
            raise RuntimeError("MindIE 1025 端口不可用，停止压测")

        run_aisbench(concurrency, "warmup", WARMUP_PROMPTS)
        stable_results = []

        for run_index in range(1, stable_runs_for(concurrency) + 1):
            run_result = run_aisbench(
                concurrency,
                f"run{run_index}",
                stable_prompts_for(concurrency),
            )
            stable_results.append(run_result)

            if run_index == 1 and previous_point is not None:
                throughput_regressed = (
                    run_result["throughput_requests_s"]
                    < previous_point["throughput_requests_s"]
                )
                p99_surge = (
                    run_result["p99_latency_ms"]
                    >= previous_point["p99_latency_ms"] * OVERLOAD_P99_GROWTH
                )
                if throughput_regressed and p99_surge:
                    overload_marker.write_text(
                        "throughput regressed and P99 increased by at least 25%\n",
                        encoding="utf-8",
                    )
                    print(f"[STOP] 并发 {concurrency} 判定为过载点")
                    return {
                        "status": "overload_stop",
                        "concurrency": concurrency,
                    }

        previous_point = pooled_point_result(stable_results)
        print(
            f"[DONE] c={concurrency} "
            f"throughput={previous_point['throughput_requests_s']:.4f} requests/s "
            f"P99={previous_point['p99_latency_ms']:.2f} ms"
        )

    return {"status": "complete", "concurrency": CONCURRENCIES[-1]}


sweep_result = run_sweep()
sweep_result


## 10. 汇总正式轮结果并保存 CSV

预热轮不进入 CSV。吞吐量取同一并发点各正式轮的均值；P95/P99 将各轮逐请求 E2EL 合并后重新计算，不对各轮分位数做简单平均。检测为过载的并发点允许只有首轮，其余并发点必须达到计划轮数。


In [ ]:
def collect_formal_runs():
    collected = []

    for concurrency in CONCURRENCIES:
        concurrency_root = OUTPUT_ROOT / f"c{concurrency}"
        overload = (concurrency_root / "OVERLOAD_STOP").exists()
        expected_runs = stable_runs_for(concurrency)
        run_results = []

        for run_index in range(1, expected_runs + 1):
            phase = f"run{run_index}"
            result_dir = completed_result_dir(concurrency_root / phase)
            if result_dir is None:
                continue
            result = read_run_result(result_dir)
            result.update(
                {
                    "concurrency": concurrency,
                    "phase": phase,
                    "point_status": "overload_stop" if overload else "complete",
                }
            )
            run_results.append(result)

        if not run_results:
            continue
        if not overload and len(run_results) != expected_runs:
            raise RuntimeError(
                f"并发 {concurrency} 只有 {len(run_results)}/{expected_runs} 个正式轮结果"
            )
        collected.extend(run_results)

        if overload:
            break

    if not collected:
        raise RuntimeError("没有可汇总的 AISBench 正式轮结果")
    return collected


formal_runs = collect_formal_runs()
len(formal_runs)


In [ ]:
raw_rows = []
for result in formal_runs:
    raw_rows.append(
        {
            "concurrency": result["concurrency"],
            "run": result["phase"],
            "request_count": result["request_count"],
            "throughput_requests_s": result["throughput_requests_s"],
            "avg_latency_ms": result["avg_latency_ms"],
            "p95_latency_ms": result["p95_latency_ms"],
            "p99_latency_ms": result["p99_latency_ms"],
            "output_token_throughput_tokens_s": result[
                "output_token_throughput_tokens_s"
            ],
            "success_requests": result["success_requests"],
            "failed_requests": result["failed_requests"],
            "point_status": result["point_status"],
            "result_dir": str(result["result_dir"]),
        }
    )

raw_df = pd.DataFrame(raw_rows).sort_values(["concurrency", "run"])
raw_df.to_csv(RAW_CSV, index=False, encoding="utf-8")
display(raw_df)


In [ ]:
summary_rows = []
for concurrency, group_df in raw_df.groupby("concurrency", sort=True):
    group_results = [
        result for result in formal_runs
        if result["concurrency"] == concurrency
    ]
    pooled_latencies = [
        latency
        for result in group_results
        for latency in result["request_latencies_ms"]
    ]
    throughput_values = group_df["throughput_requests_s"].tolist()

    summary_rows.append(
        {
            "concurrency": int(concurrency),
            "throughput_requests_s": statistics.fmean(throughput_values),
            "throughput_std_requests_s": (
                statistics.stdev(throughput_values)
                if len(throughput_values) > 1
                else 0.0
            ),
            "avg_latency_ms": statistics.fmean(pooled_latencies),
            "p95_latency_ms": float(np.percentile(pooled_latencies, 95)),
            "p99_latency_ms": float(np.percentile(pooled_latencies, 99)),
            "stable_runs": len(group_results),
            "requests_per_run": int(group_df["request_count"].iloc[0]),
            "pooled_requests": len(pooled_latencies),
            "failed_requests": int(group_df["failed_requests"].sum()),
            "point_status": group_df["point_status"].iloc[0],
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("concurrency")
numeric_columns = [
    "throughput_requests_s",
    "throughput_std_requests_s",
    "avg_latency_ms",
    "p95_latency_ms",
    "p99_latency_ms",
]
summary_df[numeric_columns] = summary_df[numeric_columns].round(6)
summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8")
display(summary_df)


## 11. 绘制性能图表

绘图单元只画必要曲线、坐标轴、标题和图例，不添加文字注释、区域说明、最大值标记、过载标记或业务 SLO 说明。三张图均使用汇总 CSV 重新读取数据，确保图表与交付 CSV 一致。


In [ ]:
plt.rcParams.update(
    {
        "font.sans-serif": [
            "PingFang SC",
            "Hiragino Sans GB",
            "Noto Sans CJK SC",
            "Source Han Sans SC",
            "SimHei",
            "Arial Unicode MS",
            "DejaVu Sans",
        ],
        "axes.unicode_minus": False,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 140,
        "savefig.dpi": 200,
    }
)

plot_df = pd.read_csv(SUMMARY_CSV).sort_values("concurrency")
x = plot_df["concurrency"].to_numpy()
throughput = plot_df["throughput_requests_s"].to_numpy()
p95 = plot_df["p95_latency_ms"].to_numpy()
p99 = plot_df["p99_latency_ms"].to_numpy()

CHART_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.plot(x, throughput, color="#C7000B", marker="o", linewidth=2.2)
ax.set_xscale("log", base=2)
ax.set_xticks(x, [str(value) for value in x])
ax.set_title("并发数—吞吐量")
ax.set_xlabel("AISBench 最大并发请求数（batch_size）")
ax.set_ylabel("请求吞吐量（requests/s）")
ax.grid(axis="y", linewidth=0.7, alpha=0.35)
fig.tight_layout()
fig.savefig(CHART_DIR / "01_concurrency_throughput.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.plot(x, p95, color="#5B5B5B", marker="o", linewidth=2.0, label="P95 E2EL")
ax.plot(x, p99, color="#C7000B", marker="o", linewidth=2.2, label="P99 E2EL")
ax.set_xscale("log", base=2)
ax.set_xticks(x, [str(value) for value in x])
ax.set_title("并发数—P95/P99 端到端延迟")
ax.set_xlabel("AISBench 最大并发请求数（batch_size）")
ax.set_ylabel("E2EL 延迟（ms）")
ax.grid(axis="y", linewidth=0.7, alpha=0.35)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(CHART_DIR / "02_concurrency_tail_latency.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.plot(throughput, p99, color="#C7000B", marker="o", linewidth=2.2)
ax.set_title("吞吐量—P99 端到端延迟")
ax.set_xlabel("请求吞吐量（requests/s）")
ax.set_ylabel("P99 E2EL 延迟（ms）")
ax.grid(axis="y", linewidth=0.7, alpha=0.35)
fig.tight_layout()
fig.savefig(CHART_DIR / "03_throughput_p99_latency.png", bbox_inches="tight")
plt.show()


## 12. 输出验收

最后检查两个 CSV 和三张图是否存在、是否非空，并确认正式结果中没有失败请求。若压测因过载规则停止，过载点仍会保留在 CSV 和曲线中，但图中不会附加任何说明文字。


In [ ]:
expected_outputs = [
    RAW_CSV,
    SUMMARY_CSV,
    CHART_DIR / "01_concurrency_throughput.png",
    CHART_DIR / "02_concurrency_tail_latency.png",
    CHART_DIR / "03_throughput_p99_latency.png",
]

for path in expected_outputs:
    ok = path.is_file() and path.stat().st_size > 0
    print(f"[{'OK' if ok else 'FAIL'}] {path}")

lab["outputs_ok"] = all(
    path.is_file() and path.stat().st_size > 0
    for path in expected_outputs
)
lab["failed_requests"] = int(summary_df["failed_requests"].sum())
lab["benchmark_ok"] = lab["outputs_ok"] and lab["failed_requests"] == 0

print("失败请求数:", lab["failed_requests"])
print("实验验收:", "PASS" if lab["benchmark_ok"] else "FAIL")


## 13. 小结

本实验完成了 AISBench 配置生成、可恢复的并发扫描、过载停止、正式轮结果汇总、CSV 保存和三张性能曲线绘制。

复核结果时重点确认：

1. 实际最大并发是否等于计划中的 <code>batch_size</code>；
2. 所有正式轮的失败请求数是否为 0；
3. 并发增加时吞吐量何时停止明显增长；
4. P95/P99 E2EL 是否在高并发区快速上升；
5. 图表是否仅包含必要曲线，没有额外文字说明或特殊点位标记。
